## SQS — the queue (standard vs. FIFO)

SQS is a fully managed **message queue**. A producer calls `SendMessage`; a consumer calls `ReceiveMessage` (up to 10 at once), does the work, then `DeleteMessage`. If it **never deletes** (crash, timeout, error), the message reappears after the **visibility timeout** and another consumer retries it — the heart of the **at-least-once** guarantee. While being processed, a message is hidden so two workers don't both run it; a consumer can extend with `ChangeMessageVisibility` if the work runs long. Messages are up to **256 KB** and retained up to **14 days**.

Two flavours, chosen almost entirely by **ordering**. **Standard** — best-effort order, at-least-once (occasional duplicates), effectively **unlimited throughput** — fits most async work (image processing, email, log shipping); the consumer must be **idempotent** (safe to process the same message twice), which is good design anyway. **FIFO** — **strict order per message group** and **exactly-once** processing, at ~**300 msg/s** (3,000 batched); the queue name must end `.fifo`. Its two concepts: a **Message Group ID** (messages in a group are strictly ordered; different groups run in parallel) and a **deduplication ID** (drops duplicates within a 5-minute window).

The default is **Standard + idempotent consumers**; reach for **FIFO** only when order or exactly-once genuinely matters.